In [10]:
import sys
import os

sys.path.append(os.path.abspath("../src"))

In [11]:
import pandas as pd 
import numpy as np 
from preprocessing.my_standard_scaler import MyStandardScaler
from preprocessing.my_minmax_scaler import My_MinMax_Scaler
from preprocessing.my_one_hot_encoder import MyOneHotEncoder
from preprocessing.my_ordinal_encoder import MyOrdinalEncoder

In [16]:
class MyColumnTransformer :

    def __init__(self,transformers):
        """
        Parameters
        ----------
        transformers : list of tuples 
            Format:
            [
                ("name",transformer_object,["column1","column2"]),
                ...
            ]
        """
        self.transformers = transformers 
        self.fitted_transformers = []

    def fit(self,X):
        if not isinstance(X,pd.DataFrame):
            raise TypeError(
                "Input must be a pandas DataFrame."
            )
        self.fitted_transformers = []

        for name , transformer , columns in self.transformers:
            transformer.fit(X[columns])
            self.fitted_transformers.append(
                (name , transformer , columns)
            )

        return self 

    def transform(self,X):
        if not isinstance(X,pd.DataFrame):
            raise TypeError(
                "Input must be a pandas DataFrame."
            )
        if not self.fitted_transformers:
            raise ValueError(
                "call fit before transform"
            )
        transform_parts = []
        for name , transformer , columns in self.fitted_transformers:
            transformed = transformer.transform(X[columns])
            transform_parts.append(transformed)
        
        return pd.concat(transform_parts , axis =1)

    def fit_transform(self,X):
        self.fit(X)
        return self.transform(X)



In [17]:
df = pd.DataFrame(
    {
      "Age":[25,30,45],
      "Salary":[40000,60000,90000],
      "City":["Delhi","Mumbai","Delhi"],
      "Education":["Bachelor's","Master's","PhD"]  
    }
)
preprocessor = MyColumnTransformer(
    [
        ("age_scaler",MyStandardScaler(),["Age"]),
        ("Salary_scaler",My_MinMax_Scaler(),["Salary"]),
        ("City_encoder",MyOneHotEncoder(),["City"]),
        ("Education_ordinal",MyOrdinalEncoder(
            categories = {
                "Education":["Bachelor's","Master's","PhD"] 
            }
        ),["Education"])
    ]
)
preprocessor.fit_transform(df)

,Age,Salary,City_Delhi,City_Mumbai,Education
0,-0.980581,0.0,1,0,0
1,-0.392232,0.4,0,1,1
2,1.372813,1.0,1,0,2
